<a href="https://colab.research.google.com/github/tauqeer-jetcom/ML_PROJECTS_1-61023748031/blob/main/Recommendation_System_Using_Tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
import pandas as pd
import numpy as np

df = pd.read_csv("netflix_content.csv")
df

,Title,Available Globally?,Release Date,Hours Viewed,Language Indicator,Content Type
0,The Night Agent: Season 1,Yes,2023-03-23,"81,21,00,000",English,Show
1,Ginny & Georgia: Season 2,Yes,2023-01-05,"66,51,00,000",English,Show
2,The Glory: Season 1 // 더 글로리: 시즌 1,Yes,2022-12-30,"62,28,00,000",Korean,Show
3,Wednesday: Season 1,Yes,2022-11-23,"50,77,00,000",English,Show
4,Queen Charlotte: A Bridgerton Story,Yes,2023-05-04,"50,30,00,000",English,Movie
...,...,...,...,...,...,...
24807,We Are Black and British: Season 1,No,NaN,"1,00,000",English,Show
24808,Whitney Cummings: Can I Touch It?,Yes,2019-07-30,"1,00,000",English,Movie
24809,Whitney Cummings: Jokes,No,2022-07-26,"1,00,000",English,Movie
24810,"Whose Vote Counts, Explained: Limited Series",Yes,2020-09-28,"1,00,000",English,Movie


In [38]:
df.size

148872

In [39]:
df.columns

Index(['Title', 'Available Globally?', 'Release Date', 'Hours Viewed',
       'Language Indicator', 'Content Type'],
      dtype='object')

In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24812 entries, 0 to 24811
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Title                24812 non-null  object
 1   Available Globally?  24812 non-null  object
 2   Release Date         8166 non-null   object
 3   Hours Viewed         24812 non-null  object
 4   Language Indicator   24812 non-null  object
 5   Content Type         24812 non-null  object
dtypes: object(6)
memory usage: 1.1+ MB


In [41]:
df.describe()

,Title,Available Globally?,Release Date,Hours Viewed,Language Indicator,Content Type
count,24812,24812,8166,24812,24812,24812
unique,19158,2,1783,889,6,2
top,La Reina del Sur: Season 3,No,2020-03-20,"1,00,000",English,Movie
freq,2,17162,28,4046,17268,14104


In [44]:
import pandas as pd
df = pd.read_csv("netflix_content.csv")

df["Hours Viewed"] = df["Hours Viewed"].str.replace(',','',regex=False).astype('int64')

df.dropna(subset=['Title'],inplace=True)
df.drop_duplicates(subset=['Title'],inplace=True)

df['Content_ID'] = df.reset_index().index.astype('int32')

df['Language_ID'] = df['Language Indicator'].astype('category').cat.codes
df['ContentType_ID'] = df['Content Type'].astype('category').cat.codes

df[['Content_ID', 'Title', 'Hours Viewed', 'Language_ID', 'ContentType_ID']].head()

,Content_ID,Title,Hours Viewed,Language_ID,ContentType_ID
0,0,The Night Agent: Season 1,812100000,0,1
1,1,Ginny & Georgia: Season 2,665100000,0,1
2,2,The Glory: Season 1 // 더 글로리: 시즌 1,622800000,3,1
3,3,Wednesday: Season 1,507700000,0,1
4,4,Queen Charlotte: A Bridgerton Story,503000000,0,0


In [50]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# Correctly get the vocabulary size for each embedding
num_content_ids = df['Content_ID'].max() + 1
num_language_ids = df['Language_ID'].max() + 1
num_content_type_ids = df['ContentType_ID'].max() + 1

content_input = layers.Input(shape=(1,), dtype=tf.int32, name='content_id')
language_input = layers.Input(shape=(1,), dtype=tf.int32, name='language_id')
type_input = layers.Input(shape=(1,), dtype=tf.int32, name='content_type')

content_embedding = layers.Embedding(input_dim=num_content_ids, output_dim=32)(content_input)
language_embedding = layers.Embedding(input_dim=num_language_ids, output_dim=8)(language_input)
content_type_embedding = layers.Embedding(input_dim=num_content_type_ids, output_dim=4)(type_input)

content_vec = layers.Flatten()(content_embedding)
language_vec = layers.Flatten()(language_embedding)
content_type_vec = layers.Flatten()(content_type_embedding)

combined= layers.Concatenate()([content_vec, language_vec, content_type_vec])
x = layers.Dense(64, activation='relu')(combined)
x = layers.Dense(32, activation='relu')(x)
# The output layer should predict one of the content IDs, so its dimension should be num_content_ids
output = layers.Dense(num_content_ids, activation='softmax')(x)

model = Model(inputs=[content_input, language_input, type_input], outputs=output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ content_id          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ language_id         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ content_type        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, 1, 32)     │    613,056 │ content_id[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 1, 8)      │         48 │ language_id[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 1, 4)      │          8 │ content_type[0][… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32)        │          0 │ embedding_3[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 8)         │          0 │ embedding_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_2 (Flatten) │ (None, 4)         │          0 │ embedding_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 44)        │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0],  │
│                     │                   │            │ flatten_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      2,880 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 19158)     │    632,214 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,250,286 (4.77 MB)

 Trainable params: 1,250,286 (4.77 MB)

 Non-trainable params: 0 (0.00 B)

In [63]:
model.fit(
    x={'content_id':df['Content_ID'],
       'language_id':df['Language_ID'],
       'content_type':df['ContentType_ID']
       },
    y=df['Content_ID'],
    epochs=10,
    batch_size=64
)

Epoch 1/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.8676 - loss: 0.6760
Epoch 2/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.8928 - loss: 0.5266
Epoch 3/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.9199 - loss: 0.3993
Epoch 4/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - accuracy: 0.9394 - loss: 0.3017
Epoch 5/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.9540 - loss: 0.2258
Epoch 6/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.9657 - loss: 0.1668
Epoch 7/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 10s 23ms/step - accuracy: 0.9743 - loss: 0.1250
Epoch 8/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.9807 - loss: 0.0925
Epoch 9/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 7s 24ms/step - accuracy: 0.9870 - loss: 0.0671
Epoch 10/10
300/300 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.9915 - loss: 0.0466


In [64]:
import numpy as np

def recommend_similar(content_title, top_k=10):
  content_row = df[df['Title'].str.contains(content_title, case=False, na=False)].iloc[0]
  content_id = content_row['Content_ID']
  language_id = content_row['Language_ID']
  content_type_id = content_row['ContentType_ID']

  predictions = model.predict({
      'content_id':np.array([content_id]),
      'language_id':np.array([language_id]),
      'content_type':np.array([content_type_id])
  })

  top_indices = predictions[0].argsort()[-top_k-1:][::-1]
  recommendations = df[df['Content_ID'].isin(top_indices)]
  return recommendations[['Title','Language Indicator','Content Type','Hours Viewed']]
recommend_similar("Breaking Bad")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step


,Title,Language Indicator,Content Type,Hours Viewed
77,Breaking Bad: Season 2,English,Show,116200000
4664,Million Dollar Listing New York: Season 5,English,Show,3200000
4811,Dragon Age: Absolution: Season 1,English,Show,3000000
4891,Charmed (2018): Season 2,English,Show,2900000
5511,72 Cutest Animals: Season 1,English,Show,2300000
5588,"Roswell, New Mexico: Season 4",English,Show,2300000
6480,#blackAF: Season 1,English,Show,1600000
10032,Due South: Season 1,English,Show,500000
13735,Rake (2010): Season 3,English,Show,200000
14497,A Queen Is Born: Season 1 // Nasce uma Rainha:...,English,Show,100000
